# Preparación de datos — Forecasting de demanda eléctrica en Ecuador

Este notebook realiza el preprocesamiento completo del conjunto de datos de demanda
eléctrica, desde la lectura del archivo fuente hasta la generación del dataset
final listo para modelado.

**Fuente de datos:** Kaggle — *Electrical Power Data: Equatorial Regions (Ecuador)*  
🔗 https://www.kaggle.com/datasets/erikfmndez/electrical-power-data-equatorial-regions-ecuador

El archivo fuente `Data_All2.csv` contiene **23 columnas y 882.570 filas** con registros
de demanda energética de Ecuador entre 2018 y 2022.

**Alcance de este notebook:**
- Filtrar el período agosto 2021 — noviembre 2022
- Seleccionar únicamente el alimentador **Sur de Quito** (subestación Quito, alimentador Sur)
- Convertir unidades de kW a MW
- Completar la serie temporal horaria con interpolación
- Incorporar temperatura horaria desde Open-Meteo (San Rafael, Valle de los Chillos)
- Agregar variables de calendario
- Exportar el dataset final a CSV

## 1. Librerías

Librerías utilizadas para el preprocesamiento: `pandas` y `numpy` para manipulación
de datos, `requests` para la descarga de temperatura desde la API de Open-Meteo,
y `feature_engine` para la generación de variables de calendario.

In [1]:
import numpy as np
import pandas as pd
import requests
from feature_engine.datetime import DatetimeFeatures
import warnings
warnings.filterwarnings('ignore')

print(f"Versión pandas : {pd.__version__}")
print(f"Versión numpy  : {np.__version__}")

Versión pandas : 2.3.3
Versión numpy  : 2.2.6


## 2. Lectura del dataset fuente

Se carga el archivo `Data_All2.csv` con separador `;`. Se eliminan espacios en
blanco en los nombres de columnas para evitar errores de referencia en pasos
posteriores.

In [2]:
# Lectura del archivo fuente
# ==============================================================================
df = pd.read_csv(
    "../data/Data_All2.csv",
    sep=";",
    dayfirst=True
)

# Eliminar espacios en nombres de columnas
df.columns = df.columns.str.strip()

print(f"Shape          : {df.shape}")
print(f"Columnas       : {df.columns.tolist()}")
print(f"Período        : {df['datetime'].min()} → {df['datetime'].max()}")
print()
df.info()

Shape          : (882570, 23)
Columnas       : ['datetime', 'year', 'region', 'substation', 'primary_feeder', 'vln_a', 'vln_b', 'vln_c', 'vll_avg', 'kw_a', 'kw_b', 'kw_c', 'kvar_a', 'kvar_b', 'kvar_c', 'i_a', 'i_b', 'i_c', 'kw_tot', 'kvar_tot', 'kva_tot', 'latitude', 'longitude']
Período        : 1/1/2018 0:00 → 9/9/2022 9:00

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 882570 entries, 0 to 882569
Data columns (total 23 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   datetime        882570 non-null  object 
 1   year            882570 non-null  int64  
 2   region          882570 non-null  object 
 3   substation      882570 non-null  object 
 4   primary_feeder  882570 non-null  object 
 5   vln_a           882570 non-null  object 
 6   vln_b           882570 non-null  object 
 7   vln_c           882570 non-null  object 
 8   vll_avg         882570 non-null  float64
 9   kw_a            882570 non-null  object 
 10  kw_

## 3. Filtrado y selección de variables

Se filtran los registros según tres criterios:
1. **Período temporal**: agosto 2021 — noviembre 2022
2. **Subestación**: Quito
3. **Alimentador**: Sur

Se conservan únicamente las columnas `datetime` (renombrada a `Time`) y
`kw_tot` (renombrada a `Demand`), que contiene la demanda eléctrica en kW.
Los datos se ordenan cronológicamente y se reinicia el índice.

In [3]:
# Filtrado temporal y por alimentadora
# ==============================================================================
df["datetime"] = pd.to_datetime(df["datetime"], dayfirst=True)

fecha_ini = pd.to_datetime("01/08/2021 00:00", dayfirst=True)
fecha_fin = pd.to_datetime("30/11/2022 23:00", dayfirst=True)

datos = df[
    (df["datetime"] >= fecha_ini) &
    (df["datetime"] <= fecha_fin) &
    (df["substation"]     == "Quito") &
    (df["primary_feeder"] == "Sur")
].copy()

# Renombrar columnas y seleccionar solo las necesarias
datos = datos.rename(columns={"datetime": "Time", "kw_tot": "Demand"})
datos = datos[["Time", "Demand"]].copy()
datos = datos.sort_values(by="Time").reset_index(drop=True)

print(f"Registros filtrados : {len(datos)}")
print(f"Período             : {datos['Time'].min()} → {datos['Time'].max()}")
print()
print(datos.describe())

Registros filtrados : 13830
Período             : 2021-08-01 00:00:00 → 2022-11-30 23:00:00

                                Time        Demand
count                          13830  13830.000000
mean   2022-03-01 17:08:58.828633600   1773.261439
min              2021-08-01 00:00:00    639.696000
25%              2021-09-29 07:48:45   1391.337250
50%              2022-02-15 11:30:00   1817.414500
75%              2022-07-09 21:45:00   2065.535500
max              2022-11-30 23:00:00   3094.039000
std                              NaN    387.154396


## 4. Conversión de unidades y preparación del índice temporal

Se realizan tres transformaciones:
1. **Conversión kW → MW**: división por 1.000 y redondeo a 3 decimales
2. **Establecer índice temporal**: la columna `Time` pasa a ser el índice del DataFrame
3. **Filtrar horas exactas**: se retienen solo registros en minuto 0 y segundo 0
4. **Eliminar duplicados**: en caso de registros duplicados en el mismo instante, se suman sus valores

In [4]:
# Establecer índice temporal, convertir unidades y limpiar duplicados
# ==============================================================================
datos = datos.set_index("Time").sort_index()

# Convertir kW → MW
datos["Demand"] = (datos["Demand"] / 1000).round(3)

# Retener solo registros en horas exactas (minuto=0, segundo=0)
datos = datos[(datos.index.minute == 0) & (datos.index.second == 0)].copy()

# Sumar valores duplicados en el mismo instante horario
datos = datos.groupby(datos.index).agg({"Demand": "sum"})

print(f"Registros tras limpieza : {len(datos)}")
print(f"Período                 : {datos.index.min()} → {datos.index.max()}")
print(f"Demanda mínima (MW)     : {datos['Demand'].min():.3f}")
print(f"Demanda máxima (MW)     : {datos['Demand'].max():.3f}")
print(f"Demanda media (MW)      : {datos['Demand'].mean():.3f}")
print()
datos.head()

Registros tras limpieza : 11670
Período                 : 2021-08-01 00:00:00 → 2022-11-30 23:00:00
Demanda mínima (MW)     : 0.640
Demanda máxima (MW)     : 3.094
Demanda media (MW)      : 1.783



,Demand
Time,
2021-08-01 00:00:00,1.488
2021-08-01 01:00:00,1.341
2021-08-01 02:00:00,1.285
2021-08-01 03:00:00,1.229
2021-08-01 04:00:00,1.181


## 5. Verificación de integridad de la serie horaria

Se comprueba que la serie no tenga huecos temporales. Se genera el índice
horario teórico completo para el período de estudio y se compara con el
índice real del DataFrame. Las horas faltantes se identifican explícitamente
para decidir la estrategia de imputación.

In [5]:
# Verificar integridad de la serie horaria
# ==============================================================================
fecha_inicio_serie = datos.index.min()
fecha_fin_serie    = datos.index.max()

date_range_completo = pd.date_range(
    start = fecha_inicio_serie,
    end   = fecha_fin_serie,
    freq  = "h"
)

horas_faltantes = date_range_completo.difference(datos.index)

print(f"Horas esperadas  : {len(date_range_completo)}")
print(f"Horas existentes : {len(datos)}")
print(f"Horas faltantes  : {len(horas_faltantes)}")

if len(horas_faltantes) > 0:
    print(f"\nPrimeras horas faltantes:")
    print(horas_faltantes[:10])
else:
    print("\n✅ La serie horaria está completa — no hay huecos")

Horas esperadas  : 11688
Horas existentes : 11670
Horas faltantes  : 18

Primeras horas faltantes:
DatetimeIndex(['2021-08-06 07:00:00', '2021-08-06 08:00:00',
               '2021-08-06 09:00:00', '2021-08-06 10:00:00',
               '2021-08-06 12:00:00', '2021-08-06 13:00:00',
               '2022-02-06 01:00:00', '2022-02-06 02:00:00',
               '2022-02-06 03:00:00', '2022-05-12 20:00:00'],
              dtype='datetime64[ns]', freq=None)


## 6. Completar serie horaria e interpolación

Se fuerza la frecuencia horaria con `asfreq("h")`, que inserta `NaN` en las
horas faltantes identificadas en el paso anterior. A continuación se aplica
interpolación temporal (`method="time"`), que estima los valores ausentes
respetando la continuidad de la serie y el intervalo de tiempo entre observaciones.
Finalmente se redondea a 3 decimales y se verifica que no queden valores nulos.

In [6]:
# Completar serie horaria con interpolación temporal
# ==============================================================================
datos = (
    datos
    .asfreq("h")                   # inserta NaN en horas faltantes
    .interpolate(method="time")    # estima valores usando interpolación temporal
    .round(3)                      # redondea a 3 decimales
)

nan_restantes = datos["Demand"].isna().sum()
print(f"NaN tras interpolación : {nan_restantes}")
print(f"Frecuencia             : {datos.index.freq}")
print(f"Registros totales      : {len(datos)}")
print()
datos.info()

NaN tras interpolación : 0
Frecuencia             : <Hour>
Registros totales      : 11688

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 11688 entries, 2021-08-01 00:00:00 to 2022-11-30 23:00:00
Freq: h
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   Demand  11688 non-null  float64
dtypes: float64(1)
memory usage: 182.6 KB


## 7. Incorporación de temperatura horaria

Se descarga la temperatura del aire a 2 metros de altura desde la API histórica
de **Open-Meteo** (gratuita, sin clave API) para la ubicación de la alimentadora:
**San Rafael, Valle de los Chillos, Quito** (lat: -0.2517, lon: -78.4500).

La temperatura se fusiona con la serie de demanda mediante un `LEFT JOIN` sobre
el índice temporal, conservando únicamente las horas presentes en `datos`
(agosto 2021 — noviembre 2022).

In [7]:
# Descarga de temperatura horaria desde Open-Meteo
# ==============================================================================
# Coordenadas de San Rafael, Valle de los Chillos, Quito
lat, lon = -0.2517, -78.4500

url = (
    f"https://archive-api.open-meteo.com/v1/archive?"
    f"latitude={lat}&longitude={lon}"
    f"&start_date=2021-08-01&end_date=2022-11-30"
    f"&hourly=temperature_2m"
    f"&timezone=America%2FGuayaquil"
)

response  = requests.get(url)
data_meteo = response.json()

df_clima = pd.DataFrame(data_meteo["hourly"])
df_clima["time"] = pd.to_datetime(df_clima["time"])
df_clima = df_clima.set_index("time")
df_clima = df_clima.rename(columns={"temperature_2m": "Temperature"})

# Fusionar temperatura con datos de demanda (LEFT JOIN sobre índice temporal)
datos = datos.merge(
    df_clima[["Temperature"]],
    left_index  = True,   # índice de datos (fechas de demanda)
    right_index = True,   # índice de df_clima (fechas de temperatura)
    how         = "left"  # conserva todas las filas de datos
)

print(f"✅ Temperatura incorporada")
print(f"   NaN en Temperature : {datos['Temperature'].isna().sum()}")
print(f"   Temperatura mínima : {datos['Temperature'].min():.1f} °C")
print(f"   Temperatura máxima : {datos['Temperature'].max():.1f} °C")
print(f"   Temperatura media  : {datos['Temperature'].mean():.1f} °C")
print()
datos.head(3)

✅ Temperatura incorporada
   NaN en Temperature : 0
   Temperatura mínima : 4.5 °C
   Temperatura máxima : 25.2 °C
   Temperatura media  : 14.1 °C



,Demand,Temperature
Time,,
2021-08-01 00:00:00,1.488,10.4
2021-08-01 01:00:00,1.341,9.9
2021-08-01 02:00:00,1.285,9.6


## 8. Variables de calendario

Se extraen cuatro variables temporales directamente del índice datetime:
- `month`: mes del año (1–12)
- `week`: semana del año (1–52)
- `day_of_week`: día de la semana (0=lunes, 6=domingo)
- `hour`: hora del día (0–23)

Estas variables serán utilizadas como features exógenas en los modelos
XGBoost y LSTM para capturar los patrones estacionales de la demanda.

In [8]:
# Generación de variables de calendario
# ==============================================================================
features_to_extract = ['month', 'week', 'day_of_week', 'hour']

calendar_transformer = DatetimeFeatures(
    variables           = 'index',
    features_to_extract = features_to_extract,
    drop_original       = True,
)

variables_calendario = calendar_transformer.fit_transform(datos)[features_to_extract]

# Incorporar al dataset principal
datos = pd.concat([datos, variables_calendario], axis=1)

print(f"✅ Variables de calendario incorporadas")
print(f"   Columnas: {datos.columns.tolist()}")

✅ Variables de calendario incorporadas
   Columnas: ['Demand', 'Temperature', 'month', 'week', 'day_of_week', 'hour']


## 9. Validación final del dataset

Se verifica que el dataset final cumpla los requisitos para el modelado:
- Sin valores nulos
- Frecuencia horaria completa
- Columnas esperadas presentes
- Período correcto

In [9]:
# Validación final del dataset
# ==============================================================================
print("=" * 50)
print("RESUMEN DEL DATASET FINAL")
print("=" * 50)
print(f"Shape            : {datos.shape}")
print(f"Período          : {datos.index[0]}  →  {datos.index[-1]}")
print(f"Frecuencia       : {datos.index.freq}")
print(f"NaN totales      : {datos.isnull().sum().sum()}")
print(f"Columnas         : {datos.columns.tolist()}")
print("=" * 50)
print()
print(datos.describe().round(3))
print()
datos.head(5)

RESUMEN DEL DATASET FINAL
Shape            : (11688, 6)
Período          : 2021-08-01 00:00:00  →  2022-11-30 23:00:00
Frecuencia       : <Hour>
NaN totales      : 0
Columnas         : ['Demand', 'Temperature', 'month', 'week', 'day_of_week', 'hour']

          Demand  Temperature      month       week  day_of_week       hour
count  11688.000    11688.000  11688.000  11688.000    11688.000  11688.000
mean       1.783       14.121      7.269     29.694        2.994     11.500
std        0.389        3.444      3.298     14.321        2.004      6.922
min        0.640        4.500      1.000      1.000        0.000      0.000
25%        1.400       11.700      5.000     18.000        1.000      5.750
50%        1.827       13.100      8.000     33.000        3.000     11.500
75%        2.076       16.500     10.000     42.000        5.000     17.250
max        3.094       25.200     12.000     52.000        6.000     23.000



,Demand,Temperature,month,week,day_of_week,hour
Time,,,,,,
2021-08-01 00:00:00,1.488,10.4,8,30,6,0
2021-08-01 01:00:00,1.341,9.9,8,30,6,1
2021-08-01 02:00:00,1.285,9.6,8,30,6,2
2021-08-01 03:00:00,1.229,9.3,8,30,6,3
2021-08-01 04:00:00,1.181,8.6,8,30,6,4


## 10. Exportación del dataset final

Se guarda el dataset procesado en formato CSV para su uso en el notebook
de modelado (`CompareXGBoostLSTM.ipynb`). El archivo incluye el índice
temporal (`Time`) y todas las columnas procesadas.

In [11]:
# Exportar dataset final a CSV
# ==============================================================================
ruta_salida = "../data/data_2021_2022_interpolado.csv"
datos.to_csv(ruta_salida)

print(f"✅ Dataset exportado a: {ruta_salida}")
print(f"   Registros : {len(datos)}")
print(f"   Columnas  : {datos.columns.tolist()}")
print(f"   Período   : {datos.index[0]}  →  {datos.index[-1]}")

✅ Dataset exportado a: ../data/data_2021_2022_interpolado.csv
   Registros : 11688
   Columnas  : ['Demand', 'Temperature', 'month', 'week', 'day_of_week', 'hour']
   Período   : 2021-08-01 00:00:00  →  2022-11-30 23:00:00
